# mini_gpt_torch : Pre-training a 124M-Parameter Mini GPT

The GPT-2 Small model contains approximately 124 million parameters and consists of 12 Transformer layers, 12 Attention heads, and 768-dimensional embeddings. Training this model from scratch on a suitable GPU can take several hours or more, depending on the hardware, dataset size, and training configurations. While most practitioners use pre-trained checkpoints instead, implementing and training an instance from scratch provides a much deeper understanding of the inner workings of the models upon which modern products are built.


### Learning Objectives
- Fully implement the GPT-2 architecture (~124M parameters) from scratch, including token embeddings, positional embeddings, Transformer blocks, and the language model head.
- Train a GPT model on a text corpus using the next-token prediction objective and cross-entropy loss.
- Implement autoregressive text generation using temperature sampling along with top-k and top-p filtering.
- Monitor training loss curves and evaluate whether the model has learned coherent linguistic patterns.


### Problem Statement
You know what a Transformer is, you have seen its diagrams, you can cite "Attention Is All You Need," and you can draw boxes labeled Multi-Head Attention on a whiteboard.

However, none of these alone mean you know precisely what happens inside the model during text generation.

The GPT-2 Small model, using weight tying, contains 124,402,944 parameters. Every single parameter value is determined through iterative execution of a training loop: running the forward pass, computing the loss, executing the backward pass, and updating the weights. This model comprises 12 Transformer blocks, 12 attention heads per block, an embedding dimension of 768, and a vocabulary size of 50,257 tokens. Each time the model generates a token, its parameters participate in a cascade of matrix operations—a sequence of computations that ingests a series of token IDs and outputs a probability distribution over the next token.

Without implementing this process yourself, the model largely remains a black box. You can query APIs or fine-tune checkpoints, but when issues arise—such as hallucinations, degenerate repetitions, or failure to follow instructions—you lack the precise mental model required to diagnose the underlying causes.

### Core Concept
#### GPT Architecture
GPT is an autoregressive language model. The term "autoregressive" means that the model generates one token at a time, where each token depends on all previously generated tokens. The architecture consists of a stack of Transformer decoder blocks.

The complete computational path from token IDs to next-token probabilities proceeds as follows:

1. Token IDs enter the model. Input shape is `(batch_size, seq_len)`.
2. Token embedding lookup is executed. Each ID is mapped to a 768-dimensional vector. Output shape is `(batch_size, seq_len, 768)`.
3. Position embedding lookup is executed. Each position index ($0, 1, 2, \dots$) is mapped to a 768-dimensional vector, with output shape matching the previous step.
4. Token embeddings and position embeddings are element-wise summed.
5. The representation passes through 12 Transformer blocks.
6. A final LayerNorm is applied.
7. A linear projection maps the embedding dimension to the vocabulary size. Output shape is `(batch_size, seq_len, vocab_size)`.
8. Softmax is applied to convert logits into probabilities.

This is the entire model architecture: no convolutions, no recurrence. The model relies exclusively on embeddings, Attention, feed-forward networks, and LayerNorm stacked iteratively.




In [ ]:
"""
Mini GPT — PyTorch implementation exercise.

ALLOWED PyTorch APIs:
    torch tensor operations, torch.nn.Parameter, torch.nn.Linear, torch.nn.Embedding,
    torch.nn.Module, torch.autograd, torch.optim.

BANNED PyTorch APIs (you must implement these yourself):
    torch.nn.LayerNorm, torch.nn.functional.layer_norm,
    torch.nn.MultiheadAttention, torch.nn.functional.scaled_dot_product_attention,
    torch.nn.Transformer / nn.TransformerEncoderLayer / nn.TransformerDecoderLayer,
    torch.nn.functional.softmax, torch.softmax, Tensor.softmax,
    torch.nn.functional.cross_entropy, torch.nn.functional.log_softmax,
    torch.nn.CrossEntropyLoss.

Gradients are handled entirely by autograd — you never write a backward pass. Every
forward pass you write must therefore stay differentiable: build outputs from tensor
operations on the inputs, and never call .detach(), .item(), .numpy(), or wrap
anything in torch.no_grad() except where a docstring explicitly says so.

All tensors are float32 unless stated otherwise, except `token_ids`, which is
torch.long. Do not hardcode dtypes or devices inside forward passes: derive them from
the incoming tensors, so the same code runs unchanged in float64.
"""


In [30]:
import torch
import torch.nn as nn
import math
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device = {device}")

GPU is available: NVIDIA GeForce RTX 5060 Laptop GPU
device = cuda


The Transformer architecture is inherently permutation invariant. By adding `pos_embed` to `token_embed`, we inject information about the positional location of each token in the sentence into the model.

**How (Broadcasting):** The `tok_emb` tensor has dimensions `(batch_size, seq_len, embed_dim)`, and `pos_emb` has dimensions `(seq_len, embed_dim)`. PyTorch automatically broadcasts `pos_emb` across the first dimension (batch) to match the dimensions of `tok_emb` for addition.

**Device Management:** To avoid `RuntimeError` (device mismatch between GPU/CPU), we used `token_ids.device` so that the position tensor (`pos_ids`) is created on the exact same memory device where `token_ids` resides.


In [31]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len):
        """
        Token and positional embedding layer.

        Args:
            vocab_size (int): Size of the vocabulary.
            embed_dim (int): Dimensionality of embedding vectors.
            max_seq_len (int): Maximum sequence length supported by positional embeddings.

        Attributes:
            token_embed (nn.Embedding): Token embedding table. Weight shape: (vocab_size, embed_dim)
            pos_embed (nn.Embedding): Positional embedding table. Weight shape: (max_seq_len, embed_dim)
        """
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)

    def forward(self, token_ids):
        """
        Computes combined token and positional embeddings for input token sequences.

        Args:
            token_ids (torch.Tensor): Token indices, dtype torch.long.
                Shape: (batch_size, seq_len)

        Returns:
            torch.Tensor: Sum of token embeddings and the positional embeddings for
                positions 0..seq_len-1, broadcast across the batch.
                Shape: (batch_size, seq_len, embed_dim)
        """
        # input shape
        batch_size, seq_len = token_ids.shape
        
        # extract token
        tok_emb = self.token_embed(token_ids)
        
        # Creating position indices (0 to seq_len-1) and moving to the same device as the input
        pos_ids = torch.arange(seq_len, device=token_ids.device)
        pos_emb = self.pos_embed(pos_ids)
        
        return tok_emb + pos_emb


In [34]:
vocab_size = 100
embed_dim = 16
max_seq_len = 32
batch_size = 2
seq_len = 10

model = Embedding(vocab_size, embed_dim, max_seq_len).to(device)
print(model)

# random input
token_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
print(token_ids)

#Forward
output = model(token_ids)
# print(output)

assert output.shape == (batch_size, seq_len, embed_dim), f"Expected {(batch_size, seq_len, embed_dim)}, got {output.shape}"
assert output.device.type == device.type, "Output is not on the correct device"

loss = output.sum()
loss.backward()

print("Test Passed: Embedding layer is working correctly with GPU.")
print(f"Output shape: {output.shape}")
print(f"Device: {output.device}")


Embedding(
  (token_embed): Embedding(100, 16)
  (pos_embed): Embedding(32, 16)
)
tensor([[89,  0, 11, 59, 14, 82, 99, 43, 57, 82],
        [85, 35, 25, 63, 83, 40, 69, 61, 68, 17]], device='cuda:0')
Test Passed: Embedding layer is working correctly with GPU.
Output shape: torch.Size([2, 10, 16])
Device: cuda:0


The goal of Layer Normalization in neural networks, and especially in Transformers, is to stabilize the distribution of activations across layers.

The key reasons for its implementation are as follows:

**Preventing Internal Covariate Shift:**
As the network deepens, the distribution of inputs to each layer changes constantly, which slows down learning. Normalization ensures that the mean of each feature approaches 0 and its variance approaches 1.

**Scale & Shift Control:**
If we were to only normalize the data, the model's capacity to learn complex patterns might be limited. That is why we use learnable parameters, `gamma` (scale) and `beta` (shift), allowing the network to determine the optimal scale and mean for subsequent layers.

**Independence from Batch Size:**
Unlike Batch Normalization, which is dependent on the batch size, LayerNorm performs calculations independently for each sample along the feature dimension (`dim`). This property is critical for large language models and text generation (like Transformers), where sequence lengths or batch sizes may vary.



- `dim=-1`: Ensures that operations are performed exclusively along the last dimension (feature/embedding dimension). This means whether your input shape is `(batch, seq, dim)` or even `(batch, dim)`, the code works without modification.

- `keepdim=True`: Crucial. With this parameter, the output of `mean` and `var` retains the last dimension as `(..., 1)` instead of squeezing it out. This allows PyTorch to automatically broadcast the tensor across all features during normalization.
Stability: Adding `eps` to the variance before taking the square root prevents `NaN` errors in case the variance is zero.


In [35]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        """
        Layer Normalization across the feature dimension.

        Args:
            dim (int): Feature/embedding dimension to normalize.
            eps (float): Epsilon added to the variance for numerical stability.

        Attributes:
            gamma (nn.Parameter): Learnable scale, initialized to ones. Shape: (dim,)
            beta (nn.Parameter): Learnable shift, initialized to zeros. Shape: (dim,)
            eps (float): Stored epsilon value.
        """
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def forward(self, x):
        """
        Normalizes the last dimension of the input tensor and applies scale and shift.

        Args:
            x (torch.Tensor): Input tensor.
                Shape: (..., dim)

        Returns:
            torch.Tensor: Layer-normalized tensor with the same shape as the input.
                The mean and the biased variance are computed over the last axis only.
                Shape: (..., dim)
        """
        # Compute the mean along the last dimension (feature dimension)
        # `keepdim=True` is necessary to preserve dimensions for broadcasting
        mean = x.mean(dim=-1, keepdim=True)
        
        # Compute the variance (biased) along the last dimension
        # Variance = Mean of squared differences from the mean
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
        
        # Normalization
        # Add eps to prevent division by zero (numerical stability)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        
        # 4. Scale and Shift
        return x_norm * self.gamma + self.beta


In [36]:
ln = LayerNorm(dim=6).to(device)
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
                   [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]], device=device)
output = ln(x)

print("\nOutput values:\n", output)
print("\nCheck if output is on device:", output.device)


Output values:
 tensor([[[-1.4638, -0.8783, -0.2928,  0.2928,  0.8783,  1.4638],
         [-0.4472, -0.4472, -0.4472, -0.4472, -0.4472,  2.2360]]],
       device='cuda:0', grad_fn=<AddBackward0>)

Check if output is on device: cuda:0


### Multi-Head Self-Attention Implementation

This implementation represents the core of the Transformer. The logic and implementation details are as follows:

#### Purpose
*   **Projection (Q, K, V):** The input `x` (containing sequence information) is projected into three distinct spaces to simultaneously extract "Query," "Key," and "Value."
*   **Multi-Head:** Instead of a single large attention mechanism, we create multiple parallel smaller mechanisms (`num_heads`). This allows the model to attend to different relational dimensions between tokens (e.g., syntactic relationships in one head and semantic relationships in another).
*   **Causal Masking:** Since GPT is a generative model, it must not "see" future tokens. We apply masking to suppress future information.

#### Implementation Steps
1.  **Projection:** Transform `(B, T, C)` into three separate tensors for Q, K, and V.
2.  **Reshape & Transpose:** Reshape to `(B, H, T, Head_dim)` to perform matrix operations independently for each "head."
3.  **Attention Scores:** Dot product of `Q` and `K` transposed (scaled by $1/\sqrt{d_k}$).
4.  **Causal Masking:** Add `float('-inf')` to positions above the main diagonal of the score matrix.
5.  **Softmax:** Normalize scores into probability weights.
6.  **Context:** Multiply normalized weights by `V`.
7.  **Concat & Output:** Concatenate heads and project back to original dimension `C`.
```python
# 4. Apply Causal Mask
if mask is not None:
# mask must be (T, T) with 0 for allowed and -inf for masked
att = att.masked_fill(mask == 0, float('-inf'))

# 5. Softmax and weight V
att = torch.nn.functional.softmax(att, dim=-1)
y = att @ v # (B, H, T, T) @ (B, H, T, Head_dim) -> (B, H, T, Head_dim)

# 6. Concatenate heads
y = y.transpose(1, 2).contiguous().view(B, T, C)

# 7. Final output
return self.W_out(y)


In [39]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        """
        Causal Multi-Head Attention module.

        Args:
            embed_dim (int): Total dimensionality of input and output features.
            num_heads (int): Number of parallel attention heads. Must divide embed_dim.

        Attributes:
            num_heads (int): Number of attention heads.
            head_dim (int): embed_dim // num_heads.
            W_q (nn.Linear): Query projection, no bias. Weight shape: (embed_dim, embed_dim)
            W_k (nn.Linear): Key projection, no bias. Weight shape: (embed_dim, embed_dim)
            W_v (nn.Linear): Value projection, no bias. Weight shape: (embed_dim, embed_dim)
            W_out (nn.Linear): Output projection, no bias. Weight shape: (embed_dim, embed_dim)
        """
        super().__init__()
        assert embed_dim % num_heads == 0, f"embed_dim {embed_dim} not divisible by num_heads {num_heads}"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_out = nn.Linear(embed_dim, embed_dim, bias=False)
        for layer in (self.W_q, self.W_k, self.W_v, self.W_out):
            nn.init.normal_(layer.weight, mean=0.0, std=0.02)

    def forward(self, x, mask=None):
        """
        Multi-head projection, scaled dot-product attention with optional additive
        masking, and output projection. The softmax must be implemented by hand.

        Args:
            x (torch.Tensor): Input tensor.
                Shape: (batch_size, seq_len, embed_dim)
            mask (torch.Tensor, optional): Additive attention mask, added to the
                attention scores before the softmax. Allowed positions hold 0.0, masked
                positions hold a large negative value (see `causal_mask`).
                Shape: (seq_len, seq_len) or broadcastable to
                (batch_size, num_heads, seq_len, seq_len). Defaults to None (no masking).

        Returns:
            torch.Tensor: Attention output after the heads are recombined and passed
                through W_out.
                Shape: (batch_size, seq_len, embed_dim)
        """
        B, T, C = x.shape  # Batch, Seq_len, Embed_dim

        # Computing Projections (Q, K, V)
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        # Reshaping and Transposing into Heads
        # Output: (B,num_heads, T, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Computing Scaled Dot-Product Attention
        # (B, H, T, D) @ (B, H, D, T) -> (B, H, T, T)
        scale = 1.0 / math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) * scale

        # Applying Causal Mask (if present)
        if mask is not None:
            # Mask must have dimensions (1, 1, T, T) or be broadcastable
            att = att.masked_fill(mask == 0, float('-inf'))

        # Softmax and multiplication by V
        att = F.softmax(att, dim=-1)
        y = att @ v  # (B, H, T, T) @ (B, H, T, D) -> (B, H, T, D)

        # 6. Concatenate Heads
        # Return to (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)


        return self.W_out(y)


### Feed-Forward Network

In addition to the Attention mechanism, which identifies relationships between different tokens, Transformers require a space to process features within each token and learn more complex non-linear relationships.

*   **First Layer (fc1):** Expands the feature dimensions (ff_dim is typically 4x embed_dim) to provide the model with higher capacity to learn complex patterns.
*   **Activation Function (ReLU):** Adds non-linearity to the network (without it, the entire network would effectively be a simple linear layer).
*   **Second Layer (fc2):** Projects the dimensions back to the original size (embed_dim) to allow integration with the rest of the model.

---

### Operations

*   **First Layer (`self.fc1(x)`):** The layer weights with dimensions `(ff_dim, embed_dim)` are applied to the input. Given an input of shape `(B, T, embed_dim)`, PyTorch applies this to the last dimension, resulting in an output of `(B, T, ff_dim)`.
*   **Non-linearity (`F.relu`):** Sets negative values to zero and keeps positive values unchanged.
*   **Second Layer (`self.fc2(x)`):** Compresses the values from the larger `ff_dim` back to `embed_dim`, ensuring the final output has the same dimensions as the input `(B, T, embed_dim)`.

---
### Optimization and Architectural Notes

1. **nn.ReLU usage:**
   It is common practice in PyTorch to define layers (like `nn.ReLU`) in `__init__` and call them in `forward`. However, using the functional API (`F.relu`) for activation layers without learnable parameters is standard and widely used (e.g., in nanoGPT).

2. **Alignment with Original Architecture:**
   The original GPT architecture uses `GELU` instead of `ReLU`. If aiming for exact alignment with OpenAI's implementation, `torch.nn.functional.gelu(x)` is preferred. However, `ReLU` is sufficient for learning core concepts.

3. **Performance:**
   The implementation is efficient. PyTorch handles these operations effectively, including in-place memory management on the GPU.


In [41]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim):
        """
        Position-wise Feed-Forward Network (MLP).

        Args:
            embed_dim (int): Model embedding feature dimension.
            ff_dim (int): Hidden feature dimension of the expansion layer.

        Attributes:
            fc1 (nn.Linear): Expansion layer. Weight shape: (ff_dim, embed_dim), bias: (ff_dim,)
            fc2 (nn.Linear): Contraction layer. Weight shape: (embed_dim, ff_dim), bias: (embed_dim,)
        """
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, ff_dim)
        self.fc2 = nn.Linear(ff_dim, embed_dim)
        for layer in (self.fc1, self.fc2):
            nn.init.normal_(layer.weight, mean=0.0, std=0.02)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        """
        Two-layer feed-forward transformation with a ReLU activation in between.

        Args:
            x (torch.Tensor): Input hidden states.
                Shape: (..., embed_dim)

        Returns:
            torch.Tensor: Transformed features, projected up to ff_dim and back down.
                Shape: (..., embed_dim)
        """
        # Transforming `embed_dim` to `ff_dim`
        x = self.fc1(x)
        
        # Non-linear Activation
        # OpenAi use `GELU` but we use `ReLU`
        x = torch.nn.functional.relu(x)
        
        # Contraction: Returning to `embed_dim`
        x = self.fc2(x)
        
        return x